# Tensor Decomposition
*Thomas Dooms*

In tutorial 1, we decomposed the bilinear interaction tensor using eigendecomposition. This gives us interpretable features *per output class*, but there are some issues with this. The most important one is orthogonality; you might have overlapping structures in input space which behave differently. The second is somewhat of a consequence where eigendecompositions often yield "superposed" features; we might hope that a 6 decomposes into a few edge-detectors but that generally doesn't happen. If the model shares structure across classes, eigendecomposition can't see it.

Here we take a different approach. Instead of decomposing per class, we decompose the *full* third-order tensor into a set of shared **neurons**, each with explicit input patterns and output weights. The result is a set of interpretable building blocks that the model combines to classify digits.

### Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import plotly.express as px
from plotly.subplots import make_subplots

from image import Model, MNIST
from image.sparse import Model as Sparse
from kornia.augmentation import RandomGaussianNoise

device = "cuda:0"
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)

### Training the original model

As in tutorial 1, we train a bilinear model with Gaussian noise augmentation to prevent overfitting on individual pixels.

In [ ]:
model = Model.from_config(epochs=20).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.4))

### From eigendecomposition to tensor decomposition

Recall that the bilinear model computes $\text{output}_c = \sum_{i,j} B_{c,i,j}\, x_i\, x_j$, where $B$ is the third-order interaction tensor. Eigendecomposition slices $B$ along the class axis and decomposes each slice independently.

We can instead factorize the *entire* tensor at once into a new bilinear layer:
$$B_{c,i,j} \approx \sum_{r=1}^{R} L_{i,r}\, R_{j,r}\, D_{c,r}$$

Each component $r$ is a **neuron** with three parts:
- $L_r \in \mathbb{R}^{784}$, the left input pattern
- $R_r \in \mathbb{R}^{784}$, the right input pattern  
- $D_r \in \mathbb{R}^{10}$, the output weights over classes

The neuron's activation on input $x$ is $(L_r^\top x)(R_r^\top x)$. Using the identity $ab = \tfrac{1}{4}(a+b)^2 - \tfrac{1}{4}(a-b)^2$, this becomes:
$$\tfrac{1}{4}\big((L_r + R_r)^\top x\big)^2 - \tfrac{1}{4}\big((L_r - R_r)^\top x\big)^2$$

So each neuron naturally decomposes into a **positive pattern** $L_r + R_r$ (matching it increases activation) and a **negative pattern** $L_r - R_r$ (matching it decreases activation). These are directly analogous to eigenvectors with positive and negative eigenvalues.

### Fitting the decomposition

We fit a new bilinear layer by maximizing the cosine similarity between the original interaction tensor and our approximation. The `Sparse` model stores $L$, $R$, and $D$ as learnable parameters and is optimized with Muon.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=64).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    loss = 1 - sparse.similarity(model)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"loss: {loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

### Visualizing neurons

Each neuron has a positive pattern ($L_r + R_r$), a negative pattern ($L_r - R_r$), and output weights ($D_r$). The `decompose` method returns these normalized and sorted by importance (the product of the factor norms).

In [ ]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

In [ ]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))

### Exercises

The goal of this part is to implement a decomposition that is more in line with our requirements. This is where you try to find what kinds of priors and structure work well. The code above provides a skeleton with the essentials: the `Sparse` model, the reconstruction loop, and the visualization. There's no right answer and this is a hard task to get right.

### A spatial (Gaussian-process) prior

The vanilla `Sparse` decomposition treats the 784 input pixels as an *unordered bag* — nothing tells it that pixel $(5,5)$ is adjacent to $(5,6)$. That missing spatial structure is exactly what produces the salt-and-pepper texture above.

We inject it with a **Gaussian-process prior** on every component's input patterns. Define an RBF kernel over pixel positions, $K_{ij} = \exp\!\big(-\lVert p_i - p_j \rVert^2 / 2\ell^2\big)$, and add the GP quadratic penalty to the reconstruction loss:

$$\mathcal{L} = \big(1 - \cos(\hat B, B)\big) \;+\; \lambda \sum_r \Big( L_{:,r}^\top K^{-1} L_{:,r} \;+\; R_{:,r}^\top K^{-1} R_{:,r}\Big)$$

A spatially rough pattern has a huge $v^\top K^{-1} v$; a smooth one is almost free. So the penalty pushes each component toward smooth, blob-like strokes without ever telling it *what* shape to take.

Two implementation notes (both matter in practice):

1. **Numerics.** A pure RBF kernel is severely ill-conditioned (effective rank $\ll 784$). `torch.linalg.inv` returns a non-symmetric, numerically degenerate matrix here, so `image/gp_sparse.py` builds `K_inv` via the Cholesky factor (`torch.cholesky_inverse`) after adding a small jitter $\epsilon I$.
2. **Scaling.** The raw penalty magnitude scales like $1/\epsilon$, so a hard-coded `gp_strength` is uninterpretable and silently over- or under-regularises. We instead *anchor* $\lambda$ to the penalty of the Sparse solution, so the GP loss term is $\approx c$ there. The knob $c$ is the actual regularisation pressure, and it stays comparable across length-scales.

In [ ]:
from image.gp_sparse import Model as GPSparse
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm


def gp_penalty_anchor(reference, lengthscale):
    """GP penalty of the (rough) Sparse solution under the given kernel.

    The raw penalty scales like 1/jitter, so a fixed gp_strength is
    uninterpretable. Anchoring lambda to this value makes the GP loss term
    roughly `c` at the Sparse solution, independent of kernel conditioning.
    """
    probe = GPSparse.from_config(rank=64, lengthscale=lengthscale).to(device)
    probe.left.data.copy_(reference.left.data)
    probe.right.data.copy_(reference.right.data)
    return probe.gp_penalty().item()


c = 0.1            # regularisation pressure: GP loss term ~= c at the Sparse solution
lengthscale = 2.0  # RBF length-scale in pixels
anchor = gp_penalty_anchor(sparse, lengthscale)
gp_strength = c / anchor

gp_sparse = GPSparse.from_config(rank=64, lengthscale=lengthscale, gp_strength=gp_strength).to(device)

optimizer = torch.optim.Muon(gp_sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    loss = (1 - gp_sparse.similarity(model)) + gp_strength * gp_sparse.gp_penalty()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"loss: {loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    gp_acc = (gp_sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"GP-Sparse accuracy: {gp_acc:.3f}  (lambda={gp_strength:.2e}, anchor={anchor:.1f})")

### Visualizing GP-Sparse neurons

The analogue of the salt-and-pepper plot. With the spatial prior the positive ($L_r + R_r$) and negative ($L_r - R_r$) patterns should be smooth, blob-like strokes instead of high-frequency noise.

In [ ]:
plus, minus, down, sigma = gp_sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

In [ ]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))

### Metrics

Three numbers tell the story. **Reconstruction similarity** drops only slightly (we trade a little fit for structure). **Test accuracy** stays essentially unchanged — the smoothing is almost free. **Spatial total variation** — the mean absolute difference between neighbouring pixels of the normalized components — drops by roughly a quarter: the salt-and-pepper texture thinning out.

The $\sigma$-weighted TV column weights each component by its importance to the reconstruction. It comes out almost identical to the plain mean, which is the useful finding: the smoothing is **uniform across components**, not an artifact of averaging over unimportant near-zero-$\sigma$ noise directions. The reduction is genuine but moderate — the GP prior cleans the texture without forcing the strokes into unrecognisable blobs.

In [ ]:
import pandas as pd


def _column_tv(cols):
    """Per-column total variation of (784, n) patterns viewed as 28x28 images."""
    imgs = cols.T.reshape(-1, 28, 28)
    return ((imgs[:, 1:, :] - imgs[:, :-1, :]).abs().mean((1, 2))
            + (imgs[:, :, 1:] - imgs[:, :, :-1]).abs().mean((1, 2)))


def spatial_total_variation(components):
    """Mean total variation of (784, k) components treated as 28x28 images."""
    return _column_tv(components).mean().item()


def report(decomp):
    plus, minus, _, sigma = decomp.decompose()
    plus, minus, sigma = plus.cpu(), minus.cpu(), sigma.cpu()
    comps = torch.cat([plus, minus], dim=1)
    # Aggregate TV is diluted: ~half the 64 components are near-zero-sigma noise
    # in *both* models and barely move. Weighting by sigma (importance to the
    # reconstruction) tracks what you actually see in the top-component plot.
    w = (sigma / sigma.sum())
    tv_p, tv_m = _column_tv(plus), _column_tv(minus)
    return {
        "reconstruction_sim": decomp.similarity(model).item(),
        "test_accuracy": (decomp(test.x).argmax(-1) == test.y).float().mean().item(),
        "spatial_TV": spatial_total_variation(comps),
        "spatial_TV_sigma_weighted": (0.5 * ((w * tv_p).sum() + (w * tv_m).sum())).item(),
    }


pd.DataFrame({"Sparse": report(sparse), "GP-Sparse": report(gp_sparse)}).T

### Length-scale sweep

The length-scale $\ell$ controls how far the smoothness prior reaches. Too small ($\ell \to 0$) and the kernel becomes the identity — the GP penalty degenerates to plain ridge regression with no spatial structure, so the components look like vanilla `Sparse`. Too large ($\ell \to \infty$) and the kernel becomes rank-one — every component is forced toward a flat constant, destroying reconstruction and accuracy. The sweep below holds the regularisation pressure $c$ fixed (re-anchoring $\lambda$ per length-scale) so the figure isolates the effect of *spatial scale*, not raw kernel conditioning. The sweet spot is in the middle.

In [ ]:
sweep = {}
for ls in [0.5, 1.0, 2.0, 4.0, 8.0]:
    lam = c / gp_penalty_anchor(sparse, ls)
    g = GPSparse.from_config(rank=64, lengthscale=ls, gp_strength=lam).to(device)
    opt = torch.optim.Muon(g.parameters(), lr=0.02, momentum=0.95)
    sch = CosineAnnealingLR(opt, T_max=200)
    torch.set_grad_enabled(True)
    for _ in range(200):
        loss = (1 - g.similarity(model)) + lam * g.gp_penalty()
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    torch.set_grad_enabled(False)
    sweep[ls] = report(g)
    print(f"ls={ls}: {sweep[ls]}")

base_metrics = report(sparse)
xs = list(sweep)
fig = make_subplots(rows=1, cols=3, subplot_titles=["spatial TV", "reconstruction sim", "test accuracy"])
for j, key in enumerate(["spatial_TV", "reconstruction_sim", "test_accuracy"]):
    fig.add_scatter(x=xs, y=[sweep[x][key] for x in xs], mode="lines+markers",
                    name="GP-Sparse", showlegend=(j == 0), line_color="#636efa", row=1, col=j+1)
    fig.add_scatter(x=xs, y=[base_metrics[key]] * len(xs), mode="lines",
                    name="Sparse", showlegend=(j == 0), line=dict(dash="dash", color="gray"), row=1, col=j+1)
fig.update_xaxes(type="log", title="lengthscale (px)")
fig.update_layout(width=900, height=320, template="plotly_white", margin=dict(t=30, b=0))
fig

### Activation-sparsity decomposition

The GP prior smoothed the *shape* of each component but left all 64 in use. A different criterion, from **Attribution-based Parameter Decomposition** (Braun et al. 2025, [arXiv:2501.14926](https://arxiv.org/abs/2501.14926)): a good decomposition should be *sparsely used* — for any single input only a few components should fire.

Each component's activation on input $x$ is $a_r(x) = (L_r^\top x)(R_r^\top x)$. We add an L1 penalty on the mean absolute activation over the training data, pushing components to specialise:

$$\mathcal{L} = \big(1 - \cos(\hat B, B)\big) \;+\; \lambda\, \mathbb{E}_{x}\big[\, \overline{|a_r(x)|} \,\big]$$

Two things make it work. **Unit-norm constraint:** after every step we project each column of $L,R$ to unit norm and absorb the scale into $D$ (`normalize_()`); the CPD is multilinear so this preserves the tensor exactly, but it stops the model from trivially evading the L1 by shrinking $L,R$ and inflating $D$. **Anchoring:** $\lambda = c/\text{anchor}$, where the anchor is the activation density of the gauge-fixed `Sparse` solution, so the knob $c$ is an interpretable pressure (same idea as GP-Sparse).

The optimiser stays Muon — despite the non-smooth L1, it trains stably here (no NaNs or loss spikes once $L,R$ are kept unit-norm).

In [ ]:
from image.act_sparse import Model as ActSparse

# Anchor lambda to the (gauge-fixed) Sparse solution's activation density so c
# is an interpretable pressure -- same trick as the GP-Sparse anchor.
sparse_ref = ActSparse.from_config(rank=64).to(device)
sparse_ref.left.data.copy_(sparse.left.data)
sparse_ref.right.data.copy_(sparse.right.data)
sparse_ref.down.data.copy_(sparse.down.data)
sparse_ref.normalize_()

gen = torch.Generator().manual_seed(0)
anchor_batch = train.x[torch.randperm(len(train.x), generator=gen)[:1024]].flatten(start_dim=1)
act_anchor = sparse_ref.activation_penalty(anchor_batch).item()

c = 0.1
act_sparse = ActSparse.from_config(rank=64, act_sparsity=c).to(device)
optimizer = torch.optim.Muon(act_sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    xb = train.x[torch.randperm(len(train.x))[:1024]].flatten(start_dim=1)
    loss = (1 - act_sparse.similarity(model)) + c * act_sparse.activation_penalty(xb) / act_anchor

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    act_sparse.normalize_()  # unit-norm L,R (scale -> D): blocks L1 scale-evasion
    scheduler.step()
    pbar.set_description(f"loss: {loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    acc = (act_sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"ActSparse accuracy: {acc:.3f}  (c={c}, anchor={act_anchor:.4f})")

### Visualizing ActSparse neurons

The same `l+r` / `l-r` / `logits` grid as before. The activation-sparsity prior says nothing about spatial smoothness, so expect these to stay salt-and-peppery — the payoff here is *specialisation*, shown in the next figure, not visual smoothness.

In [ ]:
plus, minus, down, sigma = act_sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

In [ ]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))

### Top-activating images (the interpretability test)

The real test, borrowed from the SAE literature. For each of the top-8 components we show the 16 training images with the largest $|a_r(x)|$. If the activation-sparsity criterion produced *specialised* components, each row should be visually coherent — the same digit, or the same stroke. If a row is a jumble of unrelated digits, that component is still polysemantic.

In [ ]:
with torch.no_grad():
    s = act_sparse.left.norm(dim=0) * act_sparse.right.norm(dim=0) * act_sparse.down.norm(dim=0)
    top8 = s.argsort(descending=True)[:8].tolist()
    acts = act_sparse.activations(train.x).abs()

n_imgs = 16
fig = make_subplots(rows=8, cols=n_imgs, horizontal_spacing=0.002, vertical_spacing=0.012,
                    row_titles=[f"comp {c}" for c in top8])
for r, comp in enumerate(top8):
    idxs = acts[:, comp].topk(n_imgs).indices.tolist()
    for col, ix in enumerate(idxs):
        fig.add_heatmap(z=train.x[ix, 0].flip(0).cpu(), colorscale="Greys",
                        showscale=False, row=r+1, col=col+1)
fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_annotations(font_size=11)
fig.update_layout(width=n_imgs*46, height=8*46+10, margin=dict(l=0, r=40, b=0, t=6),
                  template="plotly_white")
fig

### c-sweep

`c` is the activation-sparsity pressure (the L1 loss term equals `c` at the Sparse solution). Higher `c` drives more components dead and lowers mean L0, at some cost to reconstruction. **L0@1%/@5%** = average number of components whose `|activation|` exceeds 1%/5% of the global max on a given input. **dead_frac** = fraction of the 64 components whose max activation over the whole dataset never reaches 1% of the global max. The `Sparse` row (gauge-fixed, no L1) is the reference. Watch for the two failure modes: everything dead (`c` too large) or no change vs Sparse (`c` too small).

In [ ]:
import pandas as pd


@torch.no_grad()
def act_report(decomp):
    A = decomp.activations(train.x).abs()
    gmax = A.max()
    per_comp_max = A.max(0).values
    p, m, _, _ = decomp.decompose()
    return {
        "reconstruction_sim": decomp.similarity(model).item(),
        "test_accuracy": (decomp(test.x).argmax(-1) == test.y).float().mean().item(),
        "L0@1%": (A > 0.01 * gmax).float().sum(1).mean().item(),
        "L0@5%": (A > 0.05 * gmax).float().sum(1).mean().item(),
        "dead_frac": (per_comp_max < 0.01 * gmax).float().mean().item(),
        "mean_TV": spatial_total_variation(torch.cat([p, m], dim=1).cpu()),
    }


rows = {"Sparse": act_report(sparse_ref)}
sweep_a = {}
for cv in [0.01, 0.03, 0.1, 0.3, 1.0]:
    a = ActSparse.from_config(rank=64, act_sparsity=cv).to(device)
    opt = torch.optim.Muon(a.parameters(), lr=0.02, momentum=0.95)
    sch = CosineAnnealingLR(opt, T_max=200)
    torch.set_grad_enabled(True)
    for _ in range(200):
        xb = train.x[torch.randperm(len(train.x))[:1024]].flatten(start_dim=1)
        L = (1 - a.similarity(model)) + cv * a.activation_penalty(xb) / act_anchor
        opt.zero_grad(); L.backward(); opt.step(); a.normalize_(); sch.step()
    torch.set_grad_enabled(False)
    sweep_a[cv] = act_report(a)
    rows[f"c={cv}"] = sweep_a[cv]
    print(f"c={cv}: {sweep_a[cv]}")

display(pd.DataFrame(rows).T)

xs = list(sweep_a)
fig = make_subplots(rows=1, cols=3, subplot_titles=["mean L0 (>1% max)", "reconstruction sim", "test accuracy"])
for j, key in enumerate(["L0@1%", "reconstruction_sim", "test_accuracy"]):
    fig.add_scatter(x=xs, y=[sweep_a[x][key] for x in xs], mode="lines+markers",
                    name="ActSparse", showlegend=(j == 0), line_color="#EF553B", row=1, col=j+1)
    fig.add_scatter(x=xs, y=[rows["Sparse"][key]] * len(xs), mode="lines",
                    name="Sparse", showlegend=(j == 0), line=dict(dash="dash", color="gray"), row=1, col=j+1)
fig.update_xaxes(type="log", title="c")
fig.update_layout(width=900, height=320, template="plotly_white", margin=dict(t=30, b=0))
fig